# 2026 FIFA World Cup Predictor

### Elo ratings, Poisson goal modelling and Monte Carlo tournament simulation

This project estimates each national team's probability of winning the 2026 FIFA World Cup. It combines:

1. **Dynamic Elo ratings** calculated from 49,477 historical international matches
2. **Poisson regression** to estimate the goals scored by each team in a matchup
3. **Monte Carlo simulation** of the expanded 48-team tournament

The notebook is designed to work in Jupyter or Google Colab. Its static results remain visible on GitHub, while the controls become interactive when the notebook is run.

> **Headline result from the original 5,000 simulations:** Argentina were favourites with a 17.60% win probability, followed by Spain at 15.12%.


## 1. Setup

Place the four CSV files in the same directory as this notebook. If you are using Google Colab, the fallback uploader will prompt you when files are missing.


In [ ]:
# Uncomment in a fresh environment if required:
# %pip install pandas numpy scipy statsmodels matplotlib seaborn ipywidgets

from pathlib import Path
from collections import defaultdict
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import poisson
from IPython.display import display, Markdown, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 100)
SEED = 42


In [ ]:
REQUIRED_FILES = {
    "elo": "elo_ratings_wc2026.csv",
    "results": "results.csv",
    "fixtures": "wc_2026_fixtures.csv",
    "teams": "wc_2026_teams.csv",
}

def locate_data_directory():
    candidates = [Path("."), Path("data"), Path("upload")]
    for directory in candidates:
        if all((directory / name).exists() for name in REQUIRED_FILES.values()):
            return directory
    try:
        from google.colab import files
        print("Upload the four project CSV files.")
        files.upload()
        return Path(".")
    except ImportError as exc:
        missing = ", ".join(REQUIRED_FILES.values())
        raise FileNotFoundError(f"Could not find the required files: {missing}") from exc

DATA_DIR = locate_data_directory()
elo_history = pd.read_csv(DATA_DIR / REQUIRED_FILES["elo"], parse_dates=["snapshot_date"])
results = pd.read_csv(DATA_DIR / REQUIRED_FILES["results"], parse_dates=["date"])
fixtures = pd.read_csv(DATA_DIR / REQUIRED_FILES["fixtures"], parse_dates=["date"])
teams = pd.read_csv(DATA_DIR / REQUIRED_FILES["teams"])

print(f"Loaded {len(results):,} historical matches from {results.date.min().date()} to {results.date.max().date()}.")
print(f"Loaded {len(teams)} tournament teams and {len(fixtures)} fixtures.")
display(teams.head())


## 2. Dynamic Elo ratings

Every team begins with an Elo rating of 1500. Before each match, the expected home result is

$$E_H = \frac{1}{1 + 10^{(R_A-R_H)/400}}.$$

The observed result is encoded as

$$
W =
\begin{cases}
1 & \text{home win} \\
0.5 & \text{draw} \\
0 & \text{away win}.
\end{cases}
$$

Ratings are updated after every match using $R'_H = R_H + K(W-E_H)$. A small goal-margin multiplier rewards decisive wins while limiting the influence of any single result.


In [ ]:
def build_dynamic_elo(match_data, base_rating=1500.0, k_factor=20.0):
    ratings = defaultdict(lambda: float(base_rating))
    pre_match = []

    clean = (match_data.dropna(subset=["home_score", "away_score"])
             .sort_values("date").copy())

    for row in clean.itertuples():
        home_rating = ratings[row.home_team]
        away_rating = ratings[row.away_team]
        pre_match.append((home_rating, away_rating))

        expected_home = 1 / (1 + 10 ** ((away_rating - home_rating) / 400))
        actual_home = 1.0 if row.home_score > row.away_score else 0.5 if row.home_score == row.away_score else 0.0
        goal_margin = abs(row.home_score - row.away_score)
        effective_k = k_factor * (1 + 0.10 * goal_margin)
        change = effective_k * (actual_home - expected_home)
        ratings[row.home_team] += change
        ratings[row.away_team] -= change

    clean[["home_elo", "away_elo"]] = pre_match
    return clean, pd.Series(dict(ratings), name="elo_rating").sort_values(ascending=False)

model_data, current_elo = build_dynamic_elo(results)
top_20 = current_elo.head(20).to_frame()
display(top_20.style.format({"elo_rating": "{:.1f}"}))

fig, ax = plt.subplots(figsize=(9, 6))
top_20.sort_values("elo_rating").plot.barh(ax=ax, legend=False, color="#1f77b4")
ax.set(title="Top 20 dynamic Elo ratings", xlabel="Elo rating", ylabel="")
plt.tight_layout()
plt.show()


## 3. Poisson goal models

The goal count for each side is modelled separately. The explanatory variables are the two pre-match Elo ratings and whether the fixture was played at a neutral venue. The model is fitted to the most recent 976 completed matches, matching the original coursework run.

The signs have an intuitive interpretation: a stronger attacking side raises its expected goals, while a stronger opponent reduces them. The neutral-site indicator captures the loss of ordinary home advantage.

The original run produced the following estimates:

| Variable | Home-goals coefficient | p-value | Away-goals coefficient | p-value |
|---|---:|---:|---:|---:|
| Constant | 1.4311 | <0.001 | 1.4210 | <0.001 |
| Home Elo | 0.0018 | <0.001 | -0.0027 | <0.001 |
| Away Elo | -0.0026 | <0.001 | 0.0015 | <0.001 |
| Neutral venue | -0.1456 | 0.034 | 0.2764 | 0.002 |


In [ ]:
TRAINING_MATCHES = 976
training = model_data.tail(TRAINING_MATCHES).copy()
X = sm.add_constant(training[["home_elo", "away_elo", "neutral"]].astype(float))

home_model = sm.GLM(training["home_score"], X, family=sm.families.Poisson()).fit()
away_model = sm.GLM(training["away_score"], X, family=sm.families.Poisson()).fit()

coefficient_table = pd.DataFrame({
    "Home-goals model": home_model.params,
    "Away-goals model": away_model.params,
    "Home p-value": home_model.pvalues,
    "Away p-value": away_model.pvalues,
})
display(coefficient_table.style.format("{:.4f}"))

fit_summary = pd.DataFrame({
    "Model": ["Home goals", "Away goals"],
    "Observations": [int(home_model.nobs), int(away_model.nobs)],
    "Log likelihood": [home_model.llf, away_model.llf],
    "Deviance": [home_model.deviance, away_model.deviance],
    "Pseudo R² (Cox-Snell)": [home_model.pseudo_rsquared(), away_model.pseudo_rsquared()],
})
display(fit_summary.style.format({"Log likelihood": "{:.1f}", "Deviance": "{:.1f}", "Pseudo R² (Cox-Snell)": "{:.3f}"}))


## 4. Interactive matchup laboratory

Choose any two tournament teams to view the model's expected goals, win/draw/loss probabilities and most likely scorelines. Knockout matches that finish level are resolved using a rating-based penalty approximation in the tournament simulator, but the table below reports the 90-minute probabilities.


In [ ]:
TEAM_ALIASES = {
    "Czechia": "Czech Republic",
    "USA": "United States",
    "Türkiye": "Turkey",
}

def rating_for(team):
    return float(current_elo.get(TEAM_ALIASES.get(team, team), 1500.0))

def expected_goals(team_1, team_2, neutral=True):
    # Direct linear-predictor evaluation is much faster inside the simulator
    # than constructing a one-row DataFrame for every match.
    home_elo, away_elo = rating_for(team_1), rating_for(team_2)
    values = {"const": 1.0, "home_elo": home_elo, "away_elo": away_elo, "neutral": float(neutral)}
    lambda_1 = float(np.exp(sum(home_model.params[k] * values[k] for k in home_model.params.index)))
    lambda_2 = float(np.exp(sum(away_model.params[k] * values[k] for k in away_model.params.index)))
    return np.clip(lambda_1, 0.05, 6.0), np.clip(lambda_2, 0.05, 6.0)

def score_probability_matrix(team_1, team_2, neutral=True, max_goals=7):
    lambda_1, lambda_2 = expected_goals(team_1, team_2, neutral)
    goals = np.arange(max_goals + 1)
    matrix = np.outer(poisson.pmf(goals, lambda_1), poisson.pmf(goals, lambda_2))
    matrix /= matrix.sum()
    return lambda_1, lambda_2, matrix

def show_matchup(team_1="Argentina", team_2="Spain", neutral=True):
    if team_1 == team_2:
        print("Select two different teams.")
        return
    l1, l2, matrix = score_probability_matrix(team_1, team_2, neutral)
    p1, pdra, p2 = np.tril(matrix, -1).sum(), np.trace(matrix), np.triu(matrix, 1).sum()
    summary = pd.DataFrame({
        "Expected goals": [l1, l2],
        "90-minute win probability": [p1, p2],
    }, index=[team_1, team_2])
    display(summary.style.format({"Expected goals": "{:.2f}", "90-minute win probability": "{:.1%}"}))
    print(f"Draw probability: {pdra:.1%}")
    fig, ax = plt.subplots(figsize=(7, 5.5))
    sns.heatmap(matrix, annot=True, fmt=".1%", cmap="Blues", ax=ax, cbar=False)
    ax.set(xlabel=f"{team_2} goals", ylabel=f"{team_1} goals", title="Modelled score probabilities")
    plt.tight_layout()
    plt.show()

if WIDGETS_AVAILABLE:
    team_options = sorted(teams["team"].unique())
    widgets.interact(
        show_matchup,
        team_1=widgets.Dropdown(options=team_options, value="Argentina", description="Team 1"),
        team_2=widgets.Dropdown(options=team_options, value="Spain", description="Team 2"),
        neutral=widgets.Checkbox(value=True, description="Neutral venue"),
    )
else:
    show_matchup()


## 5. Team explorer

The historical Elo file provides an independent annual rating series. Use this control to inspect a team's trajectory, tournament group and key metadata.


In [ ]:
def explore_team(team="England"):
    meta = teams.loc[teams["team"].eq(team)].set_index("team")
    display(meta)
    history_name = TEAM_ALIASES.get(team, team)
    history = elo_history.loc[elo_history["country"].eq(history_name)].sort_values("snapshot_date")
    if history.empty:
        print("No annual Elo history is available under this exact team name.")
        return
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history["snapshot_date"], history["rating"], color="#d62728", linewidth=2)
    ax.set(title=f"{team}: annual Elo history", xlabel="Year", ylabel="Rating")
    plt.tight_layout()
    plt.show()

if WIDGETS_AVAILABLE:
    widgets.interact(
        explore_team,
        team=widgets.Dropdown(options=sorted(teams["team"].unique()), value="England", description="Team"),
    )
else:
    explore_team()


## 6. Monte Carlo tournament engine

Each simulation plays the 72 group matches, ranks teams by points, goal difference and goals scored, advances the top two sides plus the eight strongest third-placed teams, and then plays five knockout rounds. The supplied fixture file uses a simplified third-place bracket, which is retained here.

The random seed makes a run exactly reproducible. Increasing the number of simulations reduces Monte Carlo noise but takes longer.


In [ ]:
def simulate_score(team_1, team_2, rng):
    l1, l2 = expected_goals(team_1, team_2, neutral=True)
    return int(rng.poisson(l1)), int(rng.poisson(l2))

def rank_group(group_teams, group_fixtures, rng):
    table = {team: {"team": team, "points": 0, "gd": 0, "gf": 0} for team in group_teams}
    for row in group_fixtures.itertuples():
        g1, g2 = simulate_score(row.team1, row.team2, rng)
        table[row.team1]["gf"] += g1
        table[row.team1]["gd"] += g1 - g2
        table[row.team2]["gf"] += g2
        table[row.team2]["gd"] += g2 - g1
        if g1 > g2:
            table[row.team1]["points"] += 3
        elif g2 > g1:
            table[row.team2]["points"] += 3
        else:
            table[row.team1]["points"] += 1
            table[row.team2]["points"] += 1
    ranked = list(table.values())
    rng.shuffle(ranked)
    return sorted(ranked, key=lambda x: (x["points"], x["gd"], x["gf"]), reverse=True)

def knockout_winner(team_1, team_2, rng):
    g1, g2 = simulate_score(team_1, team_2, rng)
    if g1 != g2:
        return team_1 if g1 > g2 else team_2
    p_team_1 = 1 / (1 + 10 ** ((rating_for(team_2) - rating_for(team_1)) / 400))
    return team_1 if rng.random() < p_team_1 else team_2

def simulate_tournament(rng):
    group_rankings = {}
    third_place = []
    group_games = fixtures.loc[fixtures["stage"].eq("Group Stage")]
    for group, group_team_rows in teams.groupby("group"):
        ranking = rank_group(group_team_rows["team"].tolist(), group_games.loc[group_games["group"].eq(group)], rng)
        group_rankings[group] = ranking
        third_place.append(ranking[2])

    best_thirds = sorted(third_place, key=lambda x: (x["points"], x["gd"], x["gf"]), reverse=True)[:8]
    round_32 = []
    groups = sorted(group_rankings)
    for left, right in zip(groups[::2], groups[1::2]):
        round_32.extend([
            (group_rankings[left][0]["team"], group_rankings[right][1]["team"]),
            (group_rankings[right][0]["team"], group_rankings[left][1]["team"]),
        ])
    for i in range(0, 8, 2):
        round_32.append((best_thirds[i]["team"], best_thirds[i + 1]["team"]))

    survivors = [knockout_winner(a, b, rng) for a, b in round_32]
    while len(survivors) > 1:
        survivors = [knockout_winner(survivors[i], survivors[i + 1], rng) for i in range(0, len(survivors), 2)]
    return survivors[0]

def run_simulations(n_simulations=500, seed=42, progress=False):
    rng = np.random.default_rng(seed)
    wins = defaultdict(int)
    for i in range(int(n_simulations)):
        wins[simulate_tournament(rng)] += 1
        if progress and i and i % 1000 == 0:
            print(f"Completed {i:,} simulations...")
    output = (pd.Series(wins, name="wins")
              .reindex(teams["team"], fill_value=0)
              .div(n_simulations)
              .sort_values(ascending=False)
              .rename("win_probability")
              .to_frame())
    return output


In [ ]:
def simulation_dashboard(n_simulations=500, seed=42):
    fresh = run_simulations(n_simulations, seed)
    display(fresh.head(20).style.format({"win_probability": "{:.2%}"}))
    fig, ax = plt.subplots(figsize=(10, 6))
    fresh.head(20).sort_values("win_probability").plot.barh(ax=ax, legend=False, color="#2ca02c")
    ax.set(title=f"Fresh forecast: {n_simulations:,} simulations (seed {seed})", xlabel="Tournament win probability", ylabel="")
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
    plt.tight_layout()
    plt.show()
    return fresh

if WIDGETS_AVAILABLE:
    simulation_controls = widgets.interactive(
        simulation_dashboard,
        n_simulations=widgets.Dropdown(options=[100, 500, 1000, 5000], value=500, description="Simulations"),
        seed=widgets.IntText(value=42, description="Seed"),
    )
    display(simulation_controls)
else:
    fresh_forecast = simulation_dashboard()


## 7. Original 5,000-simulation benchmark

The following table records the original coursework run supplied with the project. It is kept separate from the interactive engine because rerunning a simulation with a different seed or revised modelling choices will produce slightly different probabilities.

| Rank | Team | Win probability |
|---:|---|---:|
| 1 | Argentina | 17.60% |
| 2 | Spain | 15.12% |
| 3 | France | 7.64% |
| 4 | Germany | 6.86% |
| 5 | Portugal | 5.80% |
| 6 | Brazil | 5.58% |
| 7 | England | 4.62% |
| 8 | Morocco | 4.46% |
| 9 | Croatia | 3.98% |
| 10 | Colombia | 3.34% |
| 11 | Japan | 3.12% |
| 12 | Netherlands | 3.02% |
| 13 | South Korea | 2.72% |
| 14 | Mexico | 2.16% |
| 15 | Australia | 2.02% |
| 16 | Austria | 1.92% |
| 17 | Switzerland | 1.52% |
| 18 | Algeria | 1.50% |
| 19 | Ivory Coast | 1.00% |
| 20 | Norway | 0.98% |


In [ ]:
published_probabilities = {
    "Argentina": 0.1760, "Spain": 0.1512, "France": 0.0764, "Germany": 0.0686,
    "Portugal": 0.0580, "Brazil": 0.0558, "England": 0.0462, "Morocco": 0.0446,
    "Croatia": 0.0398, "Colombia": 0.0334, "Japan": 0.0312, "Netherlands": 0.0302,
    "South Korea": 0.0272, "Mexico": 0.0216, "Australia": 0.0202, "Austria": 0.0192,
    "Switzerland": 0.0152, "Algeria": 0.0150, "Ivory Coast": 0.0100, "Norway": 0.0098,
}
published = pd.Series(published_probabilities, name="win_probability").to_frame()
display(published.style.format({"win_probability": "{:.2%}"}))

fig, ax = plt.subplots(figsize=(10, 6))
published.sort_values("win_probability").plot.barh(ax=ax, legend=False, color="#1f4e78")
ax.set(title="2026 FIFA World Cup win probabilities — original 5,000 simulations", xlabel="Win probability", ylabel="")
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
plt.tight_layout()
plt.show()


## 8. Interpretation

- **Argentina (17.60%) and Spain (15.12%) form the leading pair.** No team exceeds a one-in-five chance, illustrating the uncertainty inherent in a knockout tournament.
- **Spain's leading Elo position does not mechanically make them tournament favourites.** The simulation also reflects match-level goal uncertainty, the group draw and possible knockout routes.
- **The Poisson coefficients have the expected directional pattern.** A team's own Elo raises its goal intensity and its opponent's Elo lowers it.
- **Neutral venues matter.** The indicator adjusts the scoring processes when the usual home advantage is absent.
- **Simulation probabilities are estimates, not certainties.** They depend on the historical sample, Elo specification, Poisson assumptions, bracket design and random simulation error.

### Limitations and extensions

The model assumes conditional independence between the two teams' goal totals and does not explicitly include player availability, squad selections, travel, rest days or tactical matchups. Extensions could include a Dixon-Coles low-score correction, time-decay weights, confederation effects, market odds, calibration tests and out-of-sample backtesting.


---

### Project summary

**Tools:** Python, pandas, NumPy, statsmodels, SciPy, Matplotlib, seaborn and ipywidgets  
**Methods:** Elo rating system, Poisson GLMs, probability matrices and Monte Carlo simulation  
**Data:** International results from 1872–2026, 48-team tournament metadata and the supplied 2026 fixture structure

*This is an analytical portfolio project. It is not betting advice.*
